# Image Preprocessing Analysis
Understanding 1D vs 2D vs 3D representations

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# Load sample image
df = pd.read_csv('../data/processed/dataset_metadata.csv')
img_path = df.iloc[0]['image_path']
img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
print(f"Original shape: {img.shape}")
plt.imshow(img)
plt.title('Original Image')
plt.axis('off')
plt.show()

## 3D Representation (Height × Width × Color)

In [ ]:
# Resize to 64x64 (keep 3D)
img_3d = cv2.resize(img, (64, 64))
print(f"3D shape: {img_3d.shape}")  # (64, 64, 3)
print(f"This is 3D: Height={img_3d.shape[0]}, Width={img_3d.shape[1]}, Channels={img_3d.shape[2]}")

# Visualize each channel
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img_3d)
axes[0].set_title('Full Image (3D)')
axes[1].imshow(img_3d[:,:,0], cmap='Reds')
axes[1].set_title('Red Channel')
axes[2].imshow(img_3d[:,:,1], cmap='Greens')
axes[2].set_title('Green Channel')
axes[3].imshow(img_3d[:,:,2], cmap='Blues')
axes[3].set_title('Blue Channel')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 1D Representation (Flattened - What Random Forest Uses)

In [ ]:
# Flatten to 1D
img_1d = img_3d.flatten()
print(f"1D shape: {img_1d.shape}")  # (12288,)
print(f"This is 1D: Just a list of {len(img_1d)} numbers")
print(f"First 20 values: {img_1d[:20]}")

# Visualize as line
plt.figure(figsize=(15, 3))
plt.plot(img_1d[:1000])  # Plot first 1000 values
plt.title('Flattened Image (1D) - First 1000 pixels')
plt.xlabel('Pixel Index')
plt.ylabel('Pixel Value')
plt.show()

print("\n❌ Problem: Lost spatial information!")
print("   - Can't tell which pixels are neighbors")
print("   - Can't detect shapes or patterns")

## Comparison: What Each Dimension Sees

In [ ]:
print("1D (Flatten - Random Forest):")
print(f"  Shape: {img_1d.shape}")
print(f"  Sees: [0.2, 0.5, 0.8, 0.1, ...]")
print(f"  ❌ No spatial structure")
print(f"  ❌ Can't detect patterns")
print()
print("3D (Keep structure - CNN):")
print(f"  Shape: {img_3d.shape}")
print(f"  Sees: [[[R,G,B], [R,G,B], ...], [[R,G,B], ...]]")
print(f"  ✓ Preserves spatial structure")
print(f"  ✓ Can detect patterns (circles, spots, edges)")
print()
print("Why not 4D?")
print("  4D is for: Videos (time), 3D scans (depth)")
print("  Photos are already 3D (height × width × color)")

## Visualize Pattern Detection

In [ ]:
# Show how 3D preserves patterns
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original
axes[0].imshow(img_3d)
axes[0].set_title('Original (3D preserved)')
axes[0].axis('off')

# Edge detection (only works with 2D structure)
gray = cv2.cvtColor(img_3d, cv2.COLOR_RGB2GRAY)
edges = cv2.Canny(gray, 100, 200)
axes[1].imshow(edges, cmap='gray')
axes[1].set_title('Edge Detection (needs 2D structure)')
axes[1].axis('off')

# Flattened (can't do edge detection)
axes[2].text(0.5, 0.5, 'Flattened 1D\n❌ Cannot detect edges\n❌ Lost spatial info', 
             ha='center', va='center', fontsize=14, transform=axes[2].transAxes)
axes[2].set_title('Flattened (1D)')
axes[2].axis('off')

plt.tight_layout()
plt.show()